# A evolução do estilo de jogo na NBA (1950–2026)
**Diogo Vieira dos Santos · João Gabriel de Souza Machado · Marcelo Maiato Neves**
FGV EMAp · Análise Exploratória de Dados e Visualização · 2026.2 — *Marco 2 (entrega intermediária)*

Repositório: [github.com/Diowgo/trabalho-aedv-2026](https://github.com/Diowgo/trabalho-aedv-2026)

## 1. Introdução

Em quase oito décadas, a forma de jogar basquete na NBA mudou profundamente: o ritmo das partidas, o tipo de arremesso preferido e a eficiência dos ataques não são os mesmos de 1950, nem de 1990. Este trabalho investiga, a partir das estatísticas coletivas das equipes, como o estilo de jogo da liga se transformou e quais métricas melhor descrevem essa transformação.

Por ser a entrega intermediária, cada pergunta traz uma visualização inicial, sua leitura preliminar e o que será feito no Marco 3. As conclusões são, portanto, hipóteses de trabalho.

As medidas centrais partem da **posse**, cada vez que uma equipe controla a bola para tentar pontuar:

| Medida | Significado |
|---|---|
| Ritmo (pace) | Posses por 48 minutos. Mede quão rápido se joga. |
| ORtg | Pontos por 100 posses. Mede a eficiência do ataque, independentemente do ritmo. |
| 3PAr | Fração dos arremessos de quadra que são de 3 pontos. |
| eFG% e TS% | Aproveitamentos que ponderam o valor das cestas (eFG%) e incluem os lances livres (TS%). |
| FTr | Lances livres tentados por arremesso de quadra. Aproxima o quanto a equipe ataca o garrafão. |

A temporada é identificada pelo ano de término: a de 2025-26 aparece como 2026.

**Síntese dos achados.** A liga parece ter passado por duas ondas: uma de velocidade, nos anos 1950–60, e outra de eficiência e arremessos de longa distância, desde meados dos anos 2010. A bola de 3 pontos é a transformação mais persistente, pois reorganizou a composição da pontuação.

## 2. Dados

Usamos a base [NBA Stats (1947–present)](https://www.kaggle.com/datasets/sumitrodatta/nba-aba-baa-stats), de Sumitro Datta, disponível no Kaggle com dados do Basketball-Reference. Cada linha dos arquivos de equipe é um par equipe × temporada da fase regular. Neste marco, foram usados o `Team_Totals.csv`, com os totais da temporada (tentativas, acertos, pontos e jogos), e o `Team_Summaries.csv`, com vitórias, ritmo, ORtg, aproveitamentos e taxas. Os arquivos do lado defensivo (`Opponent_*`) ficam para o Marco 3.

Nem todas as métricas existem desde o início, o que define a janela de cada pergunta:

| Métrica | Primeira temporada disponível |
|---|---|
| Pontos, arremessos, lances livres, vitórias | 1950 |
| Ritmo, ORtg | 1951 |
| % de turnovers | 1969 |
| % de rebotes ofensivos | 1974 |
| Arremessos de 3 pontos, 3PAr | 1980 (linha de 3 pontos adotada em 1979-80) |

Na preparação, mantivemos apenas a NBA, excluindo a BAA (1947–49) e a ABA (1968–76). Calculamos as médias da liga como a soma dos totais dividida pela soma dos jogos, o que pondera corretamente temporadas com números diferentes de equipes. E, antes de 1980, tratamos as tentativas de 3 como zero, já que o arremesso não existia.

## 3. Preparação dos dados

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

CAMINHO_DADOS = next(p for p in [Path("data"), Path(".")] if (p / "Team_Totals.csv").exists())
CHAVE = ["season", "abbreviation"]
X_TEMPORADA = alt.X("season:Q", title="Temporada (ano de término)", axis=alt.Axis(format="d"),
                    scale=alt.Scale(zero=False, nice=False))
COR_DECADA = alt.Color("decada:O", title="Década", scale=alt.Scale(scheme="viridis"))

In [3]:
def carrega_tabela_nba(nome):
    df_tabela = pd.read_csv(CAMINHO_DADOS / f"{nome}.csv")
    return df_tabela[(df_tabela["lg"] == "NBA") & (df_tabela["team"] != "League Average")].copy()


db_resumos = carrega_tabela_nba("Team_Summaries")
db_totais = carrega_tabela_nba("Team_Totals")
print(db_resumos.shape, db_totais.shape)

(1693, 31) (1693, 28)


In [4]:
### Ajustando os totais da temporada

# a linha de 3 pontos só existe a partir de 1979-80: antes disso, tentativas e acertos de 3 valem 0
db_totais[["x3p", "x3pa"]] = db_totais[["x3p", "x3pa"]].fillna(0)

# tentativas de 2 pontos = tentativas de quadra - tentativas de 3
db_totais["x2pa"] = db_totais["fga"] - db_totais["x3pa"]

# acertos de 2 pontos = acertos de quadra - acertos de 3
db_totais["x2p"] = db_totais["fg"] - db_totais["x3p"]

# pontos por jogo de cada equipe = pontos / jogos
db_totais["pontos_jogo"] = db_totais["pts"] / db_totais["g"]


### Criando a tabela de equipes por temporada

# junta os resumos (ritmo, eficiência, vitórias) aos totais, usando temporada + equipe como chave
df_times = db_resumos.merge(
    db_totais[CHAVE + ["g", "pontos_jogo", "x2p", "x2pa", "x3p", "x3pa"]],
    on=CHAVE, validate="1:1")

# porcentagem de vitórias = vitórias / (vitórias + derrotas)
df_times["perc_vitorias"] = df_times["w"] / (df_times["w"] + df_times["l"])

# década de cada temporada (1957 -> 1950)
df_times["decada"] = df_times["season"] // 10 * 10


### Criando a tabela da liga por temporada

# soma dos totais de todas as equipes em cada temporada
df_soma_liga = db_totais.groupby("season")[["g", "pts", "fga", "x2p", "x2pa", "x3p", "x3pa", "ft", "fta"]].sum()

df_liga = pd.DataFrame({
    # pontos por jogo da liga = soma dos pontos / soma dos jogos
    "pontos_jogo": df_soma_liga["pts"] / df_soma_liga["g"],
    # arremessos de quadra por jogo
    "tent_quadra_jogo": df_soma_liga["fga"] / df_soma_liga["g"],
    # arremessos de 2 pontos por jogo
    "tent_2pts_jogo": df_soma_liga["x2pa"] / df_soma_liga["g"],
    # arremessos de 3 pontos por jogo
    "tent_3pts_jogo": df_soma_liga["x3pa"] / df_soma_liga["g"],
    # lances livres por jogo
    "tent_ll_jogo": df_soma_liga["fta"] / df_soma_liga["g"],
    # fração dos arremessos de quadra que são de 3 pontos, mantida como NaN antes de 1980
    "taxa_3pts": (df_soma_liga["x3pa"] / df_soma_liga["fga"]).where(df_soma_liga.index >= 1980),
    # pontos por jogo vindos de cestas de 2 (cada cesta vale 2)
    "pontos_2pts": 2 * df_soma_liga["x2p"] / df_soma_liga["g"],
    # pontos por jogo vindos de cestas de 3 (cada cesta vale 3)
    "pontos_3pts": 3 * df_soma_liga["x3p"] / df_soma_liga["g"],
    # pontos por jogo vindos de lances livres (cada acerto vale 1)
    "pontos_ll": df_soma_liga["ft"] / df_soma_liga["g"],
})

# médias entre as equipes de cada temporada; viram NaN quando nenhuma equipe tem o dado
df_medias = df_times.groupby("season").agg(
    # ritmo: posses por 48 minutos
    ritmo=("pace", "mean"),
    # eficiência ofensiva: pontos por 100 posses
    ortg=("o_rtg", "mean"),
    # aproveitamento efetivo de arremessos de quadra
    efg=("e_fg_percent", "mean"),
    # aproveitamento real, incluindo lances livres
    ts=("ts_percent", "mean"),
    # lances livres tentados por arremesso de quadra
    taxa_ll=("f_tr", "mean"),
    # porcentagem de posses que terminam em turnover
    perc_turnovers=("tov_percent", "mean"),
    # porcentagem de rebotes ofensivos capturados
    perc_reb_ofensivo=("orb_percent", "mean"))

# junta as médias à tabela da liga e traz a temporada de volta como coluna
df_liga = df_liga.join(df_medias).reset_index()

# década de cada temporada
df_liga["decada"] = df_liga["season"] // 10 * 10

df_liga.head(3)

,season,pontos_jogo,tent_quadra_jogo,tent_2pts_jogo,tent_3pts_jogo,tent_ll_jogo,taxa_3pts,pontos_2pts,pontos_3pts,pontos_ll,ritmo,ortg,efg,ts,taxa_ll,perc_turnovers,perc_reb_ofensivo,decada
0,1950,80.040998,83.084670,83.084670,0.0,33.016934,NaN,56.454545,0.0,23.586453,NaN,NaN,0.340412,0.410529,0.399118,NaN,NaN,1950
1,1951,84.059322,83.567797,83.567797,0.0,33.370056,NaN,59.584746,0.0,24.474576,97.436364,85.063636,0.356000,0.427636,0.400545,NaN,NaN,1950
2,1952,83.700000,80.828788,80.828788,0.0,33.243939,NaN,59.269697,0.0,24.430303,95.120000,86.890000,0.366300,0.438400,0.412500,NaN,NaN,1950


`df_times` tem uma linha por equipe-temporada e serve às perguntas cuja unidade é a equipe (Q5 a Q7); `df_liga` agrega a liga por temporada e atende às demais. As colunas criadas por nós têm nomes em português, e as originais da base foram mantidas. A célula seguinte mostra a partir de quando cada métrica tem dado.

In [5]:
### Primeira temporada com dado em cada métrica

# primeiro ano em que a coluna deixa de ser NaN
(df_liga.set_index("season")[["ritmo", "ortg", "perc_turnovers", "perc_reb_ofensivo", "taxa_3pts"]]
    .apply(lambda coluna: coluna.first_valid_index())
    .rename("primeira temporada com dado"))

ritmo                1951
ortg                 1951
perc_turnovers       1969
perc_reb_ofensivo    1974
taxa_3pts            1980
Name: primeira temporada com dado, dtype: int64

Os `NaN` indicam ausência da informação na base, e não erro de cálculo, e foram mantidos: imputar zero seria falso, já que as equipes cometiam turnovers e pegavam rebotes antes de 1969 e 1974. No 3 pontos, as tentativas antes de 1980 valem zero, mas a `taxa_3pts` fica como `NaN`, porque 0% sugeriria que as equipes escolhiam não arremessar de 3. O pandas ignora esses valores nas médias, e os gráficos os deixam em branco.

## 4. Q1 — Como a média de pontos por jogo evoluiu ao longo das temporadas?

Calculamos os pontos por jogo de cada equipe e a média da liga. A faixa entre o 10º e o 90º percentil das equipes mostra se o movimento é da liga inteira ou de poucos times.

In [6]:
df_faixa = df_times.groupby("season")["pontos_jogo"].quantile([0.1, 0.9]).unstack().reset_index()
df_faixa.columns = ["season", "p10", "p90"]

df_extremos = df_liga.loc[[df_liga["pontos_jogo"].idxmax(),
                           df_liga["pontos_jogo"].idxmin(),
                           df_liga[df_liga["season"] >= 1962]["pontos_jogo"].idxmin()]].copy()
df_extremos["rotulo"] = df_extremos["season"].astype(int).astype(str) + ": " + df_extremos["pontos_jogo"].round(1).astype(str)

eixo_y = alt.Y("pontos_jogo:Q", title="Pontos por jogo", scale=alt.Scale(zero=False))
grafico_faixa = alt.Chart(df_faixa).mark_area(opacity=0.2).encode(
    x=X_TEMPORADA, y=alt.Y("p10:Q", title="Pontos por jogo", scale=alt.Scale(zero=False)), y2="p90:Q")
grafico_linha = alt.Chart(df_liga).mark_line(strokeWidth=2.5).encode(
    x=X_TEMPORADA, y=eixo_y,
    tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"),
             alt.Tooltip("pontos_jogo:Q", title="Pontos por jogo", format=".1f")])
grafico_pontos = alt.Chart(df_extremos).mark_point(filled=True, size=70, color="#e45756").encode(x="season:Q", y=eixo_y)
grafico_rotulos = alt.Chart(df_extremos).mark_text(dy=-12, dx=8, fontWeight="bold").encode(
    x="season:Q", y=eixo_y, text="rotulo:N")

(grafico_faixa + grafico_linha + grafico_pontos + grafico_rotulos).properties(
    width=640, height=300,
    title="Pontos por jogo por equipe: média da liga e faixa entre o 10º e o 90º percentil das equipes")

alt.LayerChart(...)

A média sobe de cerca de 80 pontos por jogo no início dos anos 1950 (79,5 em 1954, mínimo da série) para 118,8 em 1962, salto que coincide com a adoção do relógio de arremesso de 24 segundos, em 1954-55. Depois, desce até 91,6 em 1999, temporada encurtada por lockout, e volta a 115,6 em 2026. Como a faixa acompanha a média, o movimento é coletivo.

**Marco 3:** boxplots por década, com destaque para as temporadas encurtadas (1999, 2012, 2020 e 2021).

## 5. Q2 — Como o volume, a distribuição e o aproveitamento dos arremessos mudaram?

As tentativas de 2 pontos são as de quadra menos as de 3, e o volume é a tentativa dividida pelos jogos, por equipe. Os lances livres entram como terceiro tipo. Clique na legenda para destacar uma série.

In [7]:
nomes_tipos = {"tent_2pts_jogo": "Arremessos de 2", "tent_3pts_jogo": "Arremessos de 3", "tent_ll_jogo": "Lances livres"}
df_volume = df_liga.melt(id_vars="season", value_vars=list(nomes_tipos), var_name="tipo", value_name="tentativas")
df_volume["tipo"] = df_volume["tipo"].map(nomes_tipos)

selecao_legenda = alt.selection_point(fields=["tipo"], bind="legend")
alt.Chart(df_volume).mark_line(strokeWidth=2.5).encode(
    x=X_TEMPORADA,
    y=alt.Y("tentativas:Q", title="Tentativas por jogo (por equipe)"),
    color=alt.Color("tipo:N", title="Tipo"),
    opacity=alt.condition(selecao_legenda, alt.value(1), alt.value(0.15)),
    tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"), alt.Tooltip("tipo:N", title="Tipo"),
             alt.Tooltip("tentativas:Q", title="Tentativas", format=".1f")],
).add_params(selecao_legenda).properties(width=640, height=300, title="Volume de tentativas por jogo, por tipo de arremesso")

alt.Chart(...)

Os arremessos de 2 caíram de 99,8 por jogo em 1965 para 52,1 em 2026, perto da metade, enquanto os de 3 subiram de 2,8 em 1980 para 37,0. Os lances livres caíram de 33–38 por jogo nos anos 1950–60 para cerca de 23. O salto nas tentativas de 3 em 1995 coincide com o encurtamento da linha (1994-95 a 1996-97): a regra também molda o estilo.

**Marco 3:** área empilhada 100% com a distribuição das tentativas e linhas com os percentuais de acerto (FG%, 2P%, 3P% e LL%), para testar se os arremessos de 2 passaram a ser feitos mais perto da cesta (dados por distância, a partir de 1997).

## 6. Q3 — Como a bola de três pontos foi incorporada e que impacto teve na composição do ataque?

Convertemos os acertos em pontos (2 × cestas de 2, 3 × cestas de 3 e lances livres), de modo que as três parcelas somam o total. O gráfico mostra a composição em pontos e em participação, de 1980 a 2026.

In [8]:
nomes_origens = {"pontos_2pts": "Cestas de 2", "pontos_3pts": "Cestas de 3", "pontos_ll": "Lances livres"}
df_composicao = df_liga[df_liga["season"] >= 1980].melt(
    id_vars="season", value_vars=list(nomes_origens), var_name="origem", value_name="pontos")
df_composicao["origem"] = df_composicao["origem"].map(nomes_origens)


def cria_area(empilhamento, titulo_y):
    return alt.Chart(df_composicao).mark_area().encode(
        x=X_TEMPORADA,
        y=alt.Y("pontos:Q", stack=empilhamento, title=titulo_y),
        color=alt.Color("origem:N", title="Origem dos pontos", scale=alt.Scale(range=["#4c78a8", "#f58518", "#54a24b"])),
        tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"), alt.Tooltip("origem:N", title="Origem"),
                 alt.Tooltip("pontos:Q", title="Pontos por jogo", format=".1f")],
    ).properties(width=300, height=300)


(cria_area("zero", "Pontos por jogo") | cria_area("normalize", "Participação no total")).properties(
    title="Composição da pontuação: de onde vêm os pontos (1980–2026)")

alt.HConcatChart(...)

Os pontos de 3 passam de 2,1% do total em 1980 para 34,5% em 2026 (de 2,3 para 39,9 por jogo). O ganho líquido no período é de apenas 6,3 pontos, pequeno perto das trocas internas: +37,6 de cestas de 3, −28,4 de cestas de 2 e −2,9 de lances livres. O 3 pontos não só entrou no ataque, como substituiu sobretudo os arremessos de 2.

**Marco 3:** linha da taxa de 3 (3PAr) da liga, com a faixa entre o 10º e o 90º percentil, para verificar se a adoção foi coletiva. Os dados preliminares indicam adoção gradual até 2015 e acelerada depois.

## 7. Q4 — Como o ritmo e a eficiência ofensiva se modificaram entre os períodos?

Calculamos a média de ritmo e de ORtg por temporada, ambos disponíveis desde 1951. No gráfico, cada ponto é uma temporada, ligado à seguinte, e a cor indica a década.

In [9]:
df_ritmo_efic = df_liga.dropna(subset=["ritmo", "ortg"])
base_grafico = alt.Chart(df_ritmo_efic).encode(
    x=alt.X("ritmo:Q", title="Ritmo (posses por 48 min)", scale=alt.Scale(zero=False)),
    y=alt.Y("ortg:Q", title="Eficiência ofensiva (pontos por 100 posses)", scale=alt.Scale(zero=False)),
    order="season:Q")

grafico_trilha = base_grafico.mark_line(color="#bbbbbb")
grafico_pontos = base_grafico.mark_circle(size=70).encode(
    color=COR_DECADA,
    tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"),
             alt.Tooltip("ritmo:Q", title="Ritmo", format=".1f"), alt.Tooltip("ortg:Q", title="ORtg", format=".1f")])
grafico_anos = base_grafico.transform_filter(
    alt.FieldOneOfPredicate(field="season", oneOf=[1952, 1962, 1974, 1999, 2012, 2026])
).mark_text(dx=14, dy=-8).encode(text=alt.Text("season:Q", format="d"))

(grafico_trilha + grafico_pontos + grafico_anos).properties(
    width=640, height=380, title="Trajetória da liga no plano ritmo × eficiência (cada ponto é uma temporada)")

alt.LayerChart(...)

A trajetória tem três atos. Nos anos 1950–60, a liga acelera até 126 posses por 48 minutos em 1962. Entre 1962 e 1999, desacelera e se torna mais eficiente, até o mínimo de ritmo (88,9). Desde então, sobe quase na vertical, até 115,8 pontos por 100 posses e 99,4 posses em 2026: ritmo moderado com eficiência recorde. O ritmo forma um ciclo, enquanto a eficiência cresce em quase toda a série.

**Marco 3:** linhas de ritmo e de ORtg com eixo do tempo compartilhado e uma tabela com médias por década.

## 8. Q5 — Como a pontuação por jogo se relaciona com o ritmo e a eficiência?

Como pontos por posse vezes posses por jogo dão, aproximadamente, os pontos por jogo, espera-se que ritmo e ORtg expliquem a pontuação. Para isolar essa relação da evolução histórica, comparamos as equipes dentro da mesma temporada, pelo desvio em relação à média da liga. Clique na legenda para destacar décadas.

In [10]:
df_q5 = df_times.dropna(subset=["pace", "o_rtg"]).copy()
for coluna in ["pontos_jogo", "pace", "o_rtg"]:
    df_q5[coluna + "_desvio"] = df_q5[coluna] - df_q5.groupby("season")[coluna].transform("mean")

df_q5[["pontos_jogo_desvio", "pace_desvio", "o_rtg_desvio"]].corr().round(2)

,pontos_jogo_desvio,pace_desvio,o_rtg_desvio
pontos_jogo_desvio,1.00,0.64,0.76
pace_desvio,0.64,1.00,0.01
o_rtg_desvio,0.76,0.01,1.00


In [11]:
selecao_legenda = alt.selection_point(fields=["decada"], bind="legend")


def cria_dispersao(coluna_x, titulo_x, com_legenda=True):
    base_grafico = alt.Chart(df_q5).encode(
        x=alt.X(f"{coluna_x}:Q", title=titulo_x),
        y=alt.Y("pontos_jogo_desvio:Q", title="Pontos por jogo (desvio da liga)"))
    grafico_pontos = base_grafico.mark_circle(size=35).encode(
        color=alt.Color("decada:O", title="Década", scale=alt.Scale(scheme="viridis"),
                        legend=alt.Legend() if com_legenda else None),
        opacity=alt.condition(selecao_legenda, alt.value(0.55), alt.value(0.05)),
        tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"), alt.Tooltip("team:N", title="Equipe"),
                 alt.Tooltip("pontos_jogo:Q", title="Pontos por jogo", format=".1f")])
    grafico_reta = base_grafico.transform_regression(coluna_x, "pontos_jogo_desvio").mark_line(color="black", strokeWidth=2)
    return (grafico_pontos + grafico_reta).properties(width=300, height=300)


grafico_esquerda = cria_dispersao("pace_desvio", "Ritmo (desvio da liga)", com_legenda=False)
grafico_direita = cria_dispersao("o_rtg_desvio", "ORtg (desvio da liga)").add_params(selecao_legenda)
(grafico_esquerda | grafico_direita).properties(title="Dentro de cada temporada: o que explica quem pontua mais?")

alt.HConcatChart(...)

Dentro de uma temporada, a pontuação se associa ao ritmo (r = 0,64) e à eficiência (r = 0,76), e os dois fatores são praticamente independentes entre si (r = 0,01). Juntos, parecem explicar quase toda a diferença de pontuação entre as equipes.

**Marco 3:** regressão múltipla para quantificar essa parcela e séries da liga em índice (1980 = 100), para mostrar que o pico de pontuação dos anos 1960 foi de ritmo e o de 2026, de eficiência.

## 9. Q6 — Existe relação entre o estilo de jogo e o desempenho de uma equipe na temporada?

O desempenho é a porcentagem de vitórias. O estilo é medido por 3PAr, ritmo e ORtg, cada um em z-score dentro da temporada, para que ser uma equipe de muito 3 pontos signifique o mesmo em 1985 e em 2025. A janela é 1980–2026, por causa do 3PAr.

In [12]:
nomes_metricas = {"x3p_ar": "Taxa de 3 pts (3PAr)", "pace": "Ritmo", "o_rtg": "Eficiência ofensiva (ORtg)"}
df_q6 = df_times.dropna(subset=list(nomes_metricas)).query("season >= 1980").copy()
for coluna in nomes_metricas:
    grupo = df_q6.groupby("season")[coluna]
    df_q6[coluna + "_z"] = (df_q6[coluna] - grupo.transform("mean")) / grupo.transform("std")

print(len(df_q6), "equipes-temporada")
df_q6[[coluna + "_z" for coluna in nomes_metricas]].corrwith(df_q6["perc_vitorias"]).round(2)

1314 equipes-temporada


x3p_ar_z    0.19
pace_z     -0.12
o_rtg_z     0.78
dtype: float64

In [13]:
df_q6_longo = df_q6.melt(id_vars=["season", "team", "perc_vitorias"],
                        value_vars=[coluna + "_z" for coluna in nomes_metricas], var_name="coluna", value_name="z")
df_q6_longo["metrica"] = df_q6_longo["coluna"].str.replace("_z", "").map(nomes_metricas)

base_grafico = alt.Chart().encode(
    x=alt.X("z:Q", title="Desvio em relação à liga (z-score)"),
    y=alt.Y("perc_vitorias:Q", title="% de vitórias", axis=alt.Axis(format="%")))
grafico_pontos = base_grafico.mark_circle(size=25, opacity=0.3, color="#4c78a8").encode(
    tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"), alt.Tooltip("team:N", title="Equipe"),
             alt.Tooltip("perc_vitorias:Q", title="% de vitórias", format=".1%")])
grafico_reta = base_grafico.transform_regression("z", "perc_vitorias").mark_line(color="#e45756", strokeWidth=2.5)

alt.layer(grafico_pontos, grafico_reta, data=df_q6_longo).facet(
    column=alt.Column("metrica:N", title=None, sort=list(nomes_metricas.values())),
    title="Estilo de jogo × desempenho (equipes-temporada, 1980–2026)")

alt.FacetChart(...)

A eficiência ofensiva é a variável mais associada às vitórias (r = 0,78), mas a relação é em parte mecânica, pois quem vence tende a pontuar bem por posse. O ritmo quase não se relaciona com o desempenho (r = −0,12), e a taxa de 3 tem relação positiva, porém modesta (r = 0,19). Jogar rápido não parece ser uma vantagem em si.

**Marco 3:** mapa de calor da correlação de cada métrica com as vitórias, por década, e uma variável de qualidade do elenco, já que o estilo pode ser consequência do elenco, e não causa das vitórias.

## 10. Q7 — As equipes passaram a ter estilos mais parecidos ou mais diferentes?

Cada equipe-temporada foi descrita por seis características (3PAr, ritmo, ORtg, taxa de lances livres, % de rebotes ofensivos e % de turnovers), padronizadas pela média e pelo desvio-padrão de 1980–2026. A heterogeneidade de uma temporada é a distância média das equipes ao estilo médio daquele ano, corrigida pelo número de equipes.

In [14]:
colunas_estilo = ["x3p_ar", "pace", "o_rtg", "f_tr", "orb_percent", "tov_percent"]
df_q7 = df_times.dropna(subset=colunas_estilo).query("season >= 1980")

df_z = (df_q7[colunas_estilo] - df_q7[colunas_estilo].mean()) / df_q7[colunas_estilo].std()
df_z_centrado = df_z - df_z.groupby(df_q7["season"]).transform("mean")
n_times = df_q7.groupby("season")["season"].transform("size")
df_q7 = df_q7.assign(distancia=np.sqrt((df_z_centrado ** 2).sum(axis=1) * n_times / (n_times - 1)))

df_dispersao = df_q7.groupby("season")["distancia"].mean().rename("dispersao").reset_index()
df_dispersao["dispersao_suave"] = df_dispersao["dispersao"].rolling(5, center=True, min_periods=3).mean()
df_dispersao.set_index("season").loc[[1980, 1990, 2000, 2010, 2020, 2026]].round(2)

,dispersao,dispersao_suave
season,,
1980,1.41,1.40
1990,1.68,1.57
2000,1.45,1.46
2010,1.42,1.42
2020,1.15,1.24
2026,1.41,1.33


In [15]:
grafico_bruto = alt.Chart(df_dispersao).mark_line(color="#bbbbbb", point=True).encode(
    x=X_TEMPORADA,
    y=alt.Y("dispersao:Q", title="Distância média ao estilo médio da temporada", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("season:Q", title="Temporada", format="d"), alt.Tooltip("dispersao:Q", title="Distância", format=".2f")])
grafico_suave = alt.Chart(df_dispersao).mark_line(color="#e45756", strokeWidth=3).encode(x="season:Q", y="dispersao_suave:Q")

(grafico_bruto + grafico_suave).properties(
    width=640, height=300,
    title="Heterogeneidade de estilos entre as equipes (cinza: por temporada; vermelho: média móvel de 5)")

alt.LayerChart(...)

Não há sinal de convergência sustentada: a heterogeneidade oscila entre 1,15 (2020) e 1,74 (1989), sem tendência. As equipes ficaram mais parecidas entre 2017 e 2023, e a diversidade voltou em 2024–26. A liga parece ter mudado muito mais de lugar do que de diversidade.

**Marco 3:** *small multiples* com a dispersão de cada característica e um mapa de estilos por PCA, com a trajetória do estilo médio da liga.

## 11. Q8 — Quais períodos concentram as maiores mudanças e quais métricas os caracterizam?

Cada métrica da liga foi convertida em z-score na própria série, para comparar no mesmo mapa medidas de escalas diferentes. Vermelho indica valores acima da média histórica da métrica; azul, abaixo.

In [16]:
nomes_metricas_liga = {"pontos_jogo": "Pontos por jogo", "tent_quadra_jogo": "Arremessos por jogo", "ritmo": "Ritmo",
                       "ortg": "Eficiência (ORtg)", "efg": "eFG%", "ts": "TS%", "taxa_ll": "Taxa de lances livres",
                       "taxa_3pts": "Taxa de 3 pts", "perc_turnovers": "% turnovers",
                       "perc_reb_ofensivo": "% reb. ofensivo"}

df_metricas = df_liga.set_index("season").loc[1951:, list(nomes_metricas_liga)]
df_z_metricas = (df_metricas - df_metricas.mean()) / df_metricas.std()

df_z_longo = df_z_metricas.reset_index().melt(id_vars="season", var_name="coluna", value_name="z").dropna()
df_z_longo["metrica"] = df_z_longo["coluna"].map(nomes_metricas_liga)

alt.Chart(df_z_longo).mark_rect().encode(
    x=alt.X("season:O", title="Temporada", axis=alt.Axis(values=list(range(1955, 2030, 5)), labelAngle=0)),
    y=alt.Y("metrica:N", title=None, sort=list(nomes_metricas_liga.values())),
    color=alt.Color("z:Q", title="z-score", scale=alt.Scale(scheme="redblue", domain=[-2.5, 2.5], reverse=True)),
    tooltip=[alt.Tooltip("season:O", title="Temporada"), alt.Tooltip("metrica:N", title="Métrica"),
             alt.Tooltip("z:Q", title="z-score", format=".2f")],
).properties(width=640, height=260, title="Métricas da liga padronizadas (cada linha na sua própria escala)")

alt.Chart(...)

As métricas não mudam ao mesmo tempo. Pontos, arremessos e ritmo concentram valores altos entre o fim dos anos 1950 e meados dos 1960 e depois enfraquecem. A eficiência (ORtg, eFG% e TS%) passa do azul ao vermelho de forma gradual, com avanço forte após 2015, e a taxa de 3 avança sem interrupções até os anos 2020. A taxa de 3 e a eficiência caracterizam a transformação de longo prazo; ritmo e pontuação, ciclos.

**Marco 3:** curva de intensidade de mudança com os picos rotulados (janelas preliminares: 1954–59, 1970–75 e 2015–20) e um mapa de calor da variação entre décadas.

## 12. Conclusão parcial

Os resultados preliminares sugerem que o estilo de jogo da NBA mudou em duas ondas. A primeira, nos anos 1950–60, foi de velocidade: mais posses, mais arremessos, pontuação alta e eficiência baixa. A segunda, desde meados dos anos 2010, é de eficiência e perímetro: ritmo moderado, aproveitamento recorde e uma fatia crescente dos pontos vinda do 3, a transformação mais persistente do período. A diversidade de estilos entre as equipes não mudou de forma sistemática.

**Recomendações.** Para comparar épocas, convém usar métricas por posse, como o ORtg, em vez de métricas por jogo, já que o ritmo varia em ciclos e distorce a pontuação bruta. Convém também comparar cada equipe com as da mesma temporada, como feito na Q5 e na Q6, para não confundir estilo com época.

**Próximos passos.** Além das visualizações listadas em cada seção, o Marco 3 vai incluir o lado defensivo (`Opponent_*`), os arremessos por distância (desde 1997), o controle pela qualidade do elenco e mais interatividade.

**Limitações.** Cada métrica começa em um ano diferente, e as perguntas sobre o 3 pontos cobrem apenas 1980–2026. As temporadas de 1999, 2012, 2020 e 2021 tiveram menos de 82 jogos. O ritmo e o ORtg antigos são estimados pelo Basketball-Reference, sem contagem oficial de posses. Por fim, as associações entre estilo e vitórias não são causais, e eventos como o relógio de arremesso e os lockouts são contexto, e não efeitos estimados.